## **BIA 810 - Healthcare Advanced Analytics - Final Project**

### **Problem Statement**

<div style="
    line-height: 1.8;
    text-align: justify;
    text-align-last: left;
    max-width: calc(100% - 120px);
">
The anesthesia drug market is a segment of the pharmaceutical industry that focuses on the development,
production, and distribution of medications used to induce and maintain anesthesia during medical
procedures and surgeries. The market for anesthesia drugs encompasses a wide range of pharmaceutical
products, including general anesthetics, local anesthetics, and adjunctive drugs that support the
anesthesia process. The demand for these drugs is closely tied to the healthcare industry, as the number
of surgical procedures, including both routine and complex surgeries, continues to grow worldwide.
Factors such as an aging population, increased healthcare access, and advancements in medical
technology contribute to the expansion of the anesthesia drug market.
You are a Healthcare Commercial Analytics leader working for anesthesia drugs portfolio at a big pharma
company. As an analytics leader, understanding market dynamics and making evidence-based decisions
are crucial for the success of the business. Your company has a market leading brand (Product 1- <strong>J1885</strong>) in the
anesthesia portfolio. Your company also has recently launched a variant of the same brand (Product 2- <strong>J2250</strong>) in
the market which is supposed to cannibalize your leading brand. (Market cannibalization is a loss in sales
caused by a company's introduction of a new product that displaces one of its own older products in the
market. The cannibalization of existing products need not necessarily lead to an increase in the company's
overall market share, but at least the sales growth for the new product should be at the expense of the
drop in sales of the old product.)
However, the expected cannibalization is not happening in the market. It appears that instead of your
new product capturing the dropping sales of your old product, one of your competitors (Product 3- <strong>J3010</strong>) is
rapidly gaining market share, leaving your new product (Product 2- <strong>J2250</strong>) to lose its expected market share.
</div>

In [1]:
import numpy as np
import pandas as pd

In [2]:
pd.set_option('display.max_columns', None)

### **0. Analytics-Ready Dataset Preparation**

#### 0.1 Organize Medicare Claims Data

In [3]:
claims_1_raw =  pd.read_csv("input_datasets/Medicare_Claims_data_part_1.csv")
claims_1_raw.head()

,cur_clm_uniq_id,bene_mbi_id,fac_prvdr_npi_num,clm_from_dt,clm_thru_dt,prncpl_dgns_cd,clm_pmt_amt,clm_mdcr_instnl_tot_chrg_amt,clm_line_num,clm_line_hcpcs_cd,clm_line_cvrd_pd_amt,clm_val_sqnc_num_dgns,clm_dgns_cd,clm_val_sqnc_num_prcdr,clm_prcdr_cd,clm_line_alowd_chrg_amt,clm_prvdr_spclty_cd
0,1272380,10791,1.316539e+09,11/8/2016,11/7/2016,S81811D,154.73,11182.21567,11.0,G8996,-0.04,12.0,Z951,NaN,NaN,NaN,NaN
1,631196,10107,1.033202e+09,7/21/2016,7/12/2016,S0083XA,NaN,NaN,1.0,76700,2.57,NaN,I10,NaN,NaN,8.86,69
2,1548564,10412,1.880497e+09,7/14/2018,5/20/2018,R918,NaN,NaN,1.0,C1751,53.81,NaN,I10,NaN,NaN,86.40,30
3,1427731,10934,1.655569e+09,6/19/2016,7/15/2016,I2119,705.89,20750.00517,17.0,85378,-5.91,7.0,M25552,NaN,NaN,NaN,NaN
4,428779,965,1.644309e+09,3/21/2017,8/10/2018,C4490,2884.15,18437.66432,5.0,80061,2.62,6.0,Z9049,NaN,NaN,NaN,NaN


In [4]:
claims_1 = claims_1_raw[[
    'cur_clm_uniq_id', 'bene_mbi_id', 'fac_prvdr_npi_num', 'clm_from_dt', 'clm_dgns_cd', 'clm_line_hcpcs_cd'
]].rename(
    columns={
        'cur_clm_uniq_id': 'Claim_ID',
        'bene_mbi_id': 'Patient_ID',
        'fac_prvdr_npi_num': 'HCP_ID',
        'clm_from_dt': 'Claim_Date',
        'clm_dgns_cd': 'Diagnosis_Code',
        'clm_line_hcpcs_cd': 'Procedure_Code'
    }
)
claims_1

,Claim_ID,Patient_ID,HCP_ID,Claim_Date,Diagnosis_Code,Procedure_Code
0,1272380,10791,1.316539e+09,11/8/2016,Z951,G8996
1,631196,10107,1.033202e+09,7/21/2016,I10,76700
2,1548564,10412,1.880497e+09,7/14/2018,I10,C1751
3,1427731,10934,1.655569e+09,6/19/2016,M25552,85378
4,428779,965,1.644309e+09,3/21/2017,Z9049,80061
...,...,...,...,...,...,...
199995,201316,12551,1.447581e+09,7/31/2016,I10,84132
199996,1139560,13193,1.172104e+09,3/26/2017,I10,84460
199997,1422020,11766,1.840282e+09,12/15/2018,G311,J2060
199998,276481,12995,1.676423e+09,10/2/2016,I10,80048


In [5]:
claims_2_raw =  pd.read_csv("input_datasets/Medicare_Claims_data_part_2.csv")
claims_2_raw.head()

,cur_clm_uniq_id,bene_mbi_id,fac_prvdr_npi_num,clm_from_dt,clm_thru_dt,prncpl_dgns_cd,clm_pmt_amt,clm_mdcr_instnl_tot_chrg_amt,clm_line_num,clm_line_hcpcs_cd,clm_line_cvrd_pd_amt,clm_val_sqnc_num_dgns,clm_dgns_cd,clm_val_sqnc_num_prcdr,clm_prcdr_cd,clm_line_alowd_chrg_amt,clm_prvdr_spclty_cd
0,1089764,10064,1.775153e+09,11/10/2017,3/9/2018,C439,NaN,NaN,1.0,99284,8.13,NaN,I10,NaN,NaN,7.71,30
1,677005,1174,1.585740e+09,8/25/2016,7/27/2016,R1011,NaN,NaN,4.0,86921,1.24,NaN,I10,NaN,NaN,8.55,69
2,751112,12093,1.474716e+09,2/14/2018,5/5/2018,Z951,NaN,NaN,1.0,97530,51.67,NaN,I10,NaN,NaN,100.83,8
3,569047,10773,1.300394e+09,4/8/2016,6/16/2016,D500,NaN,NaN,1.0,73522,130.49,NaN,I10,NaN,NaN,67.01,41
4,1244504,11998,1.788795e+09,3/2/2017,10/13/2016,K210,282.61,8810.400285,1.0,47563,38.53,4.0,M5032,NaN,NaN,NaN,NaN


In [6]:
claims_2 = claims_2_raw[[
    'cur_clm_uniq_id', 'bene_mbi_id', 'fac_prvdr_npi_num', 'clm_from_dt', 'clm_dgns_cd', 'clm_line_hcpcs_cd'
]].rename(
    columns={
        'cur_clm_uniq_id': 'Claim_ID',
        'bene_mbi_id': 'Patient_ID',
        'fac_prvdr_npi_num': 'HCP_ID',
        'clm_from_dt': 'Claim_Date',
        'clm_dgns_cd': 'Diagnosis_Code',
        'clm_line_hcpcs_cd': 'Procedure_Code'
    }
)
claims_2

,Claim_ID,Patient_ID,HCP_ID,Claim_Date,Diagnosis_Code,Procedure_Code
0,1089764,10064,1.775153e+09,11/10/2017,I10,99284
1,677005,1174,1.585740e+09,8/25/2016,I10,86921
2,751112,12093,1.474716e+09,2/14/2018,I10,97530
3,569047,10773,1.300394e+09,4/8/2016,I10,73522
4,1244504,11998,1.788795e+09,3/2/2017,M5032,47563
...,...,...,...,...,...,...
199995,240671,10217,1.813622e+09,5/30/2018,R197,71046
199996,779254,10678,1.685051e+09,10/16/2017,E1142,36415
199997,271289,10063,1.582805e+09,2/26/2018,I10,22851
199998,1160940,10491,1.658547e+09,8/31/2017,R197,73564


In [7]:
claims_3_raw =  pd.read_csv("input_datasets/Medicare_Claims_data_part_3.csv")
claims_3_raw.head()

,cur_clm_uniq_id,bene_mbi_id,fac_prvdr_npi_num,clm_from_dt,clm_thru_dt,prncpl_dgns_cd,clm_pmt_amt,clm_mdcr_instnl_tot_chrg_amt,clm_line_num,clm_line_hcpcs_cd,clm_line_cvrd_pd_amt,clm_val_sqnc_num_dgns,clm_dgns_cd,clm_val_sqnc_num_prcdr,clm_prcdr_cd,clm_line_alowd_chrg_amt,clm_prvdr_spclty_cd
0,1644551,11443,1.519584e+09,2/10/2018,10/20/2017,I119,NaN,NaN,1.0,83615,54.43,NaN,I10,NaN,NaN,74.65,93
1,168172,10738,1.896665e+09,11/6/2017,12/7/2017,C679,NaN,NaN,1.0,36415,72.02,NaN,I10,NaN,NaN,119.10,69
2,1076245,12980,1.615825e+09,10/27/2016,11/7/2016,R000,88.38,1081.867515,3.0,70491,44.26,NaN,I10,NaN,NaN,NaN,NaN
3,642872,11930,1.761345e+09,7/22/2018,4/12/2018,I4891,NaN,NaN,2.0,A9270,2.47,NaN,I10,NaN,NaN,9.94,69
4,1533098,1110,1.725549e+09,1/22/2016,2/6/2016,H3581,NaN,NaN,4.0,83001,0.41,NaN,I10,NaN,NaN,3.84,30


In [8]:
claims_3 = claims_3_raw[[
    'cur_clm_uniq_id', 'bene_mbi_id', 'fac_prvdr_npi_num', 'clm_from_dt', 'clm_dgns_cd', 'clm_line_hcpcs_cd'
]].rename(
    columns={
        'cur_clm_uniq_id': 'Claim_ID',
        'bene_mbi_id': 'Patient_ID',
        'fac_prvdr_npi_num': 'HCP_ID',
        'clm_from_dt': 'Claim_Date',
        'clm_dgns_cd': 'Diagnosis_Code',
        'clm_line_hcpcs_cd': 'Procedure_Code'
    }
)
claims_3

,Claim_ID,Patient_ID,HCP_ID,Claim_Date,Diagnosis_Code,Procedure_Code
0,1644551,11443,1.519584e+09,2/10/2018,I10,83615
1,168172,10738,1.896665e+09,11/6/2017,I10,36415
2,1076245,12980,1.615825e+09,10/27/2016,I10,70491
3,642872,11930,1.761345e+09,7/22/2018,I10,A9270
4,1533098,1110,1.725549e+09,1/22/2016,I10,83001
...,...,...,...,...,...,...
199995,664681,12161,1.129268e+09,2/25/2016,I10,90670
199996,933831,12684,1.146025e+09,7/15/2016,Z01812,36415
199997,456909,11037,1.818235e+09,8/2/2018,I10,A9562
199998,322749,12116,1.552952e+09,10/15/2018,I6340,93290


In [9]:
claims_4_raw =  pd.read_csv("input_datasets/Medicare_Claims_data_part_4.csv")
claims_4_raw.head()

,cur_clm_uniq_id,bene_mbi_id,fac_prvdr_npi_num,clm_from_dt,clm_thru_dt,prncpl_dgns_cd,clm_pmt_amt,clm_mdcr_instnl_tot_chrg_amt,clm_line_num,clm_line_hcpcs_cd,clm_line_cvrd_pd_amt,clm_val_sqnc_num_dgns,clm_dgns_cd,clm_val_sqnc_num_prcdr,clm_prcdr_cd,clm_line_alowd_chrg_amt,clm_prvdr_spclty_cd
0,548043,10592,1.681132e+09,6/9/2018,3/26/2016,R600,NaN,NaN,6.0,99080,-0.87,NaN,I10,NaN,NaN,8.94,30
1,1619815,10972,1.554501e+09,1/30/2018,3/26/2018,M797,6683.18,20178.55341,NaN,36415,NaN,4.0,J309,2.0,5A1955Z,NaN,NaN
2,1039752,13126,1.790419e+09,6/28/2018,8/5/2018,Z951,NaN,NaN,1.0,G8992,4.43,NaN,I10,NaN,NaN,9.79,69
3,283428,10475,1.346639e+09,6/10/2017,5/17/2017,R0600,NaN,NaN,2.0,80048,2.41,NaN,I10,NaN,NaN,8.03,69
4,901033,10338,1.649654e+09,11/22/2018,11/20/2018,K5790,NaN,NaN,4.0,83880,1.21,NaN,I10,NaN,NaN,NaN,NaN


In [10]:
claims_4 = claims_4_raw[[
    'cur_clm_uniq_id', 'bene_mbi_id', 'fac_prvdr_npi_num', 'clm_from_dt', 'clm_dgns_cd', 'clm_line_hcpcs_cd'
]].rename(
    columns={
        'cur_clm_uniq_id': 'Claim_ID',
        'bene_mbi_id': 'Patient_ID',
        'fac_prvdr_npi_num': 'HCP_ID',
        'clm_from_dt': 'Claim_Date',
        'clm_dgns_cd': 'Diagnosis_Code',
        'clm_line_hcpcs_cd': 'Procedure_Code'
    }
)
claims_4

,Claim_ID,Patient_ID,HCP_ID,Claim_Date,Diagnosis_Code,Procedure_Code
0,548043,10592,1.681132e+09,6/9/2018,I10,99080
1,1619815,10972,1.554501e+09,1/30/2018,J309,36415
2,1039752,13126,1.790419e+09,6/28/2018,I10,G8992
3,283428,10475,1.346639e+09,6/10/2017,I10,80048
4,901033,10338,1.649654e+09,11/22/2018,I10,83880
...,...,...,...,...,...,...
199995,1079928,1140,1.758394e+09,7/18/2018,R932,76856
199996,460236,12008,1.362663e+09,1/30/2016,H35373,94727
199997,1232737,11977,1.488181e+09,6/25/2018,I10,86901
199998,468338,11982,1.658223e+09,10/18/2017,I10,52214


In [11]:
claims_5_raw =  pd.read_csv("input_datasets/Medicare_Claims_data_part_5.csv")
claims_5_raw.head()

,cur_clm_uniq_id,bene_mbi_id,fac_prvdr_npi_num,clm_from_dt,clm_thru_dt,prncpl_dgns_cd,clm_pmt_amt,clm_mdcr_instnl_tot_chrg_amt,clm_line_num,clm_line_hcpcs_cd,clm_line_cvrd_pd_amt,clm_val_sqnc_num_dgns,clm_dgns_cd,clm_val_sqnc_num_prcdr,clm_prcdr_cd,clm_line_alowd_chrg_amt,clm_prvdr_spclty_cd
0,1345381,10065,1.320330e+09,4/1/2016,2/17/2016,N926,NaN,NaN,2.0,93571,77.04,NaN,I10,NaN,NaN,85.85,8
1,526682,11329,1.630461e+09,3/7/2018,1/28/2018,E780,NaN,NaN,6.0,80053,34.88,NaN,I10,NaN,NaN,81.47,6
2,1013399,11606,1.029737e+09,5/30/2018,5/9/2016,I129,NaN,NaN,2.0,84165,10.69,NaN,I10,NaN,NaN,12.53,11
3,550379,13080,1.835759e+09,2/2/2018,3/14/2018,I4891,101.84,395.427642,4.0,96375,1.24,2.0,Z951,NaN,NaN,NaN,NaN
4,874017,12778,1.043510e+09,12/1/2018,11/30/2018,D472,NaN,NaN,1.0,NaN,41.58,NaN,I10,NaN,NaN,109.53,4


In [12]:
claims_5 = claims_5_raw[[
    'cur_clm_uniq_id', 'bene_mbi_id', 'fac_prvdr_npi_num', 'clm_from_dt', 'clm_dgns_cd', 'clm_line_hcpcs_cd'
]].rename(
    columns={
        'cur_clm_uniq_id': 'Claim_ID',
        'bene_mbi_id': 'Patient_ID',
        'fac_prvdr_npi_num': 'HCP_ID',
        'clm_from_dt': 'Claim_Date',
        'clm_dgns_cd': 'Diagnosis_Code',
        'clm_line_hcpcs_cd': 'Procedure_Code'
    }
)
claims_5

,Claim_ID,Patient_ID,HCP_ID,Claim_Date,Diagnosis_Code,Procedure_Code
0,1345381,10065,1.320330e+09,4/1/2016,I10,93571
1,526682,11329,1.630461e+09,3/7/2018,I10,80053
2,1013399,11606,1.029737e+09,5/30/2018,I10,84165
3,550379,13080,1.835759e+09,2/2/2018,Z951,96375
4,874017,12778,1.043510e+09,12/1/2018,I10,NaN
...,...,...,...,...,...,...
199995,842577,27,2.094819e+09,7/15/2016,E785,J1885
199996,309023,11454,4.138986e+09,4/24/2018,I10,J3010
199997,568041,10869,5.314549e+09,1/17/2018,I10,17004
199998,100020,10784,5.599719e+09,12/23/2016,I10,J1030


In [13]:
# concatenate all 5 claims dataset together
claims_data = pd.concat(
    [claims_1, claims_2, claims_3, claims_4, claims_5],
    axis=0,
    ignore_index=True
)
claims_data

,Claim_ID,Patient_ID,HCP_ID,Claim_Date,Diagnosis_Code,Procedure_Code
0,1272380,10791,1.316539e+09,11/8/2016,Z951,G8996
1,631196,10107,1.033202e+09,7/21/2016,I10,76700
2,1548564,10412,1.880497e+09,7/14/2018,I10,C1751
3,1427731,10934,1.655569e+09,6/19/2016,M25552,85378
4,428779,965,1.644309e+09,3/21/2017,Z9049,80061
...,...,...,...,...,...,...
999995,842577,27,2.094819e+09,7/15/2016,E785,J1885
999996,309023,11454,4.138986e+09,4/24/2018,I10,J3010
999997,568041,10869,5.314549e+09,1/17/2018,I10,17004
999998,100020,10784,5.599719e+09,12/23/2016,I10,J1030


In [14]:
# check for data types and number of null value
claims_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 6 columns):
 #   Column          Non-Null Count    Dtype  
---  ------          --------------    -----  
 0   Claim_ID        1000000 non-null  int64  
 1   Patient_ID      1000000 non-null  int64  
 2   HCP_ID          999070 non-null   float64
 3   Claim_Date      1000000 non-null  str    
 4   Diagnosis_Code  999987 non-null   str    
 5   Procedure_Code  993125 non-null   str    
dtypes: float64(1), int64(2), str(3)
memory usage: 45.8 MB


In [15]:
# convert 'HCP_ID' data type to int64
claims_data['HCP_ID'] = claims_data['HCP_ID'].astype('Int64')

# convert 'Claim_Date' data type to datetime
claims_data['Claim_Date'] = pd.to_datetime(claims_data['Claim_Date'])

In [16]:
# check for duplicates
claims_data.duplicated().sum()

np.int64(0)

In [17]:
# filter claims dataset to records whose Claim_ID belong to the market brands
# filter all the records of the claims that belong to these brands and not just the records that have these Procedure_Code
market_basket = ['J1885', 'J2250', 'J3010', "J2704"]
market_basket_claim_ids = claims_data[claims_data['Procedure_Code'].isin(market_basket)]['Claim_ID'].unique()
claims_data_anesthesia = claims_data[claims_data['Claim_ID'].isin(market_basket_claim_ids)]

claims_data_anesthesia
# number of rows is in line with what is mentioned in the statement file

,Claim_ID,Patient_ID,HCP_ID,Claim_Date,Diagnosis_Code,Procedure_Code
971632,300876,12388,7348083745,2018-06-03,I10,J3010
971633,657288,11556,2202143437,2016-06-03,Z7901,J1885
971634,450553,12021,5004034153,2018-10-04,K921,99283
971635,277076,10815,4368806084,2016-12-12,I10,J3010
971636,476863,13224,5200262348,2016-06-21,Z9861,80048
...,...,...,...,...,...,...
999995,842577,27,2094818789,2016-07-15,E785,J1885
999996,309023,11454,4138986079,2018-04-24,I10,J3010
999997,568041,10869,5314548510,2018-01-17,I10,17004
999998,100020,10784,5599719066,2016-12-23,I10,J1030


#### 0.2 Load Patient Demographics Data

In [18]:
patient_demo =  pd.read_csv("input_datasets/Patient_demographics_data.csv")
patient_demo.head()

,Patient_id,Age,Gender
0,10,71,Male
1,11,56,Male
2,12,65,Female
3,13,72,Female
4,14,77,Female


In [19]:
# rename 'Patient_id' column name to match with claims dataset
patient_demo = patient_demo.rename(columns={'Patient_id': 'Patient_ID'})

In [20]:
patient_demo.info()

<class 'pandas.DataFrame'>
RangeIndex: 4508 entries, 0 to 4507
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Patient_ID  4508 non-null   int64
 1   Age         4508 non-null   int64
 2   Gender      4508 non-null   str  
dtypes: int64(2), str(1)
memory usage: 105.8 KB


#### 0.3 Load HCP Demographics Data

In [21]:
hcp_demo =  pd.read_csv("input_datasets/HCP_demographics_data.csv")
hcp_demo.head()

,HCP NPI ID,Address,City,State,ZIP Code,Specialty
0,8386928704,322 Roberts Drive Suite 888,South Shannonton,AS,67431,Neurology
1,3688956922,11734 Deanna Groves Suite 031,Leviburgh,OK,12405,Anesthesiology
2,5134290518,18686 Schwartz Streets,Shepherdstad,RI,97054,Neurology
3,6740392080,410 Woodard Drive Suite 766,East Calvinmouth,MA,92138,Gastroenterology
4,1178012810,83203 Jimenez Village Apt. 548,Griffinchester,WY,48202,Anesthesiology


In [22]:
# rename 'HCP NPI ID' column name to match with claims dataset
hcp_demo = hcp_demo.rename(columns={'HCP NPI ID': 'HCP_ID'})

In [23]:
hcp_demo.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   HCP_ID     2000 non-null   int64
 1   Address    2000 non-null   str  
 2   City       2000 non-null   str  
 3   State      2000 non-null   str  
 4   ZIP Code   2000 non-null   int64
 5   Specialty  2000 non-null   str  
dtypes: int64(2), str(4)
memory usage: 93.9 KB


In [24]:
# convert 'ZIP Code' data type to str to ensure that leading zeros are retained
hcp_demo['ZIP Code'] = hcp_demo['ZIP Code'].astype('str').str.zfill(5)
hcp_demo.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   HCP_ID     2000 non-null   int64
 1   Address    2000 non-null   str  
 2   City       2000 non-null   str  
 3   State      2000 non-null   str  
 4   ZIP Code   2000 non-null   str  
 5   Specialty  2000 non-null   str  
dtypes: int64(1), str(5)
memory usage: 93.9 KB


#### 0.4 Load Zip to Territory Mapping Data

In [25]:
zip_map =  pd.read_csv("input_datasets/Zip_to_Territory_Mapping.csv")
zip_map.head()

,Zip Code,Territory Name,Region Name
0,501,"St Louis, MO",Midwest
1,544,"St Louis, MO",Midwest
2,601,"St Louis, MO",Midwest
3,602,"St Louis, MO",Midwest
4,603,"St Louis, MO",Midwest


In [26]:
# convert 'ZIP Code' data type to str to ensure that leading zeros are retained
zip_map['Zip Code'] = zip_map['Zip Code'].astype('str').str.zfill(5)

# rename 'Zip Code' column name to match with hcp_demo
zip_map = zip_map.rename(columns={'Zip Code': 'ZIP Code'})

zip_map.info()

<class 'pandas.DataFrame'>
RangeIndex: 41683 entries, 0 to 41682
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   ZIP Code        41683 non-null  str  
 1   Territory Name  41683 non-null  str  
 2   Region Name     41683 non-null  str  
dtypes: str(3)
memory usage: 977.1 KB


In [27]:
zip_map.head()

,ZIP Code,Territory Name,Region Name
0,00501,"St Louis, MO",Midwest
1,00544,"St Louis, MO",Midwest
2,00601,"St Louis, MO",Midwest
3,00602,"St Louis, MO",Midwest
4,00603,"St Louis, MO",Midwest


#### 0.5 Load Diagnosis Code Mapping Data

In [28]:
diag_map =  pd.read_csv("input_datasets/Diagnosis_Code_Mapping.csv")
diag_map.head()

,Diagnosis Code Market,Specialty
0,A,Infectious and Parasitic Diseases
1,B,Infectious and Parasitic Diseases
2,C,Neoplasms
3,D,"Neoplasms, Blood, Blood-forming Organs"
4,E,"Endocrine, Nutritional, Metabolic"


#### 0.6 Load Procedure Code Mapping Data

In [29]:
procedure_map =  pd.read_csv("input_datasets/Procedure_Code_Mapping.csv")
procedure_map.head()

,CPT Codes,Procedure Code Category,Operative Procedure,Procedure Description,Procedure Code Descriptions
0,11008,HER,Herniorrhaphy,"Repair of inguinal, femoral, umbilical, or an...","Removal of prosthetic material or mesh, abdomi..."
1,11970,BRST,Breast surgery,Excision of lesion or tissue of breast includi...,Replacement of tissue expander(s) with permane...
2,19101,BRST,Breast surgery,Excision of lesion or tissue of breast includi...,"Biopsy of breast; open, incisional"
3,19105,BRST,Breast surgery,Excision of lesion or tissue of breast includi...,"Ablation, cryosurgical, of fibroadenoma, inclu..."
4,19110,BRST,Breast surgery,Excision of lesion or tissue of breast includi...,"Nipple exploration, with or without excision o..."
